In [2]:
import papermill as pm
import duckdb, subprocess, os, time
from joblib import Parallel, delayed
import os

In [3]:
species_list = [
        {'name':"Agelaius phoeniceus", 'group':'avian'},
        {'name':"Ammospiza leconteii", 'group':'avian'},
        {'name':"Anthus spragueii", 'group':'avian'},
        {'name':"Archilochus colubris", 'group':'avian'},
        {'name':"Buteo lineatus", 'group':'avian'},
        {'name':"Calidris melanotos", 'group':'avian'},
        {'name':"Calidris pusilla", 'group':'avian'},
        {'name':"Centronyx henslowii", 'group':'avian'},
        {'name':"Cistothorus palustris", 'group':'avian'},
        {'name':"Coccyzus americanus", 'group':'avian'},
        {'name':"Coturnicops noveboracensis", 'group':'avian'},
        {'name':"Dryocopus pileatus", 'group':'avian'},
        {'name':"Egretta caerulea", 'group':'avian'},
        {'name':"Elanoides forficatus", 'group':'avian'},
        {'name':"Empidonax virescens", 'group':'avian'},
        {'name':"Euphagus carolinus", 'group':'avian'},
        {'name':"Geothlypis formosa", 'group':'avian'},
        {'name':"Geothlypis trichas", 'group':'avian'},
        {'name':"Hylocichla mustelina", 'group':'avian'},
        {'name':"Limnothlypis swainsonii", 'group':'avian'},
        {'name':"Meleagris gallopavo", 'group':'avian'},
        {'name':"Melospiza georgiana", 'group':'avian'},
        {'name':"Melospiza melodia", 'group':'avian'},
        {'name':"Parkesia motacilla", 'group':'avian'},
        {'name':"Pluvialis dominica", 'group':'avian'},
        {'name':"Protonotaria citrea", 'group':'avian'},
        {'name':"Rallus elegans", 'group':'avian'},
        {'name':"Setophaga americana", 'group':'avian'},
        {'name':"Setophaga cerulea", 'group':'avian'},
        {'name':"Setophaga citrina", 'group':'avian'},
        {'name':"Setophaga dominica", 'group':'avian'},
        {'name':"Sphyrapicus varius", 'group':'avian'},
        {'name':"Sternula antillarum", 'group':'avian'},
        {'name':"Tringa flavipes", 'group':'avian'},
        {'name':"Vireo flavifrons", 'group':'avian'},
        {'name':"Vireo griseus", 'group':'avian'},
        {'name':"Odocoileus virginianus", 'group':'mammal'},     # white tailed deer
        {'name':"Ursus americanus", 'group':'mammal'},           # Black bear
        {'name':"Anaxyrus americanus", 'group':'herp'},          # American Toad
        {'name':"Anaxyrus fowleri", 'group':'herp'},             # Fowler's Toad
        {'name':"Gastrophryne carolinensis",'group':'herp'},     # Eastern Narrow-mouthed Toad
        {'name':"Hyla avivoca", 'group':'herp'},                 # Bird-voiced Treefrog
        {'name':"Hyla chrysoscelis", 'group':'herp'},            # Cope's Gray Treefrog
        {'name':"Hyla cinerea", 'group':'herp'},                 # Green Treefrog
        {'name':"Hyla squirella", 'group':'herp'},               # Squirrel Treefrog
        {'name':"Hyla versicolor", 'group':'herp'},              # Gray Treefrog
        {'name':"Lithobates catesbeianus", 'group':'herp'},      # American Bullfrog
        {'name':"Lithobates clamitans", 'group':'herp'},         # Bronze Frog
        {'name':"Lithobates palustris", 'group':'herp'},         # Pickerel Frog
        {'name':"Lithobates sphenocephalus", 'group':'herp'},    # Southern Leopard Frog
        {'name':"Pseudacris crucifer", 'group':'herp'},          # Spring Peeper
        {'name':"Pseudacris fouquettei", 'group':'herp'},        # Cajun Chorus Frog
        {'name':"Apalone spinifera", 'group':'herp'},            # Spiny Softshell Turtle
        {'name':"Kinosternon subrubrum", 'group':'herp'},        # Eastern Mud Turtle
        {'name':"Macrochelys temmincki", 'group':'herp'}         # Alligator Snapping Turtle
    ]
basedir = '/mnt/f/readyparams'
paramdir = os.path.join(basedir, 'param_csvs')
outputdir = os.path.join(basedir,'modelprep')
aoi = 'mav_counties_4326.parquet' #Within paramdir
jobs = 10
os.makedirs(outputdir, exist_ok=True)

In [4]:
for s in species_list:
    if isinstance(s.get("name"), str):
        s["name"] = s["name"].replace(" ", "_").lower()

def f(x):
    try:
        spoutputdir = os.path.join(outputdir,x['name'])
        os.makedirs(spoutputdir, exist_ok=True)
        print('run',x['name'])
        if jobs >1:
            subprocess.run(pm.execute_notebook(f'maxent_model.ipynb',os.path.join(spoutputdir,'{0}.ipynb'.format(x['name'])),parameters=dict(parambasedir=paramdir, baseoutputdir=basedir, sp=x['name'], spgroup=x['group'], aoi=aoi)), shell=True)
        else:
            pm.execute_notebook(f'maxent_model.ipynb',os.path.join(spoutputdir,'{0}.ipynb'.format(x['name'])),parameters=dict(parambasedir=paramdir, baseoutputdir=basedir, sp=x['name'], spgroup=x['group'], aoi=aoi))
        return (x, 'run complete')
    except Exception as e:
        return (x,'fail', e)

In [ ]:
completed = Parallel(n_jobs=10, verbose=0)(delayed(f)(species_list[x]) for x in range(len(species_list)))

run agelaius_phoeniceus
run calidris_pusilla


Executing:   0%|          | 0/23 [00:00<?, ?cell/s]

run archilochus_colubris
run cistothorus_palustris
run ammospiza_leconteii
run calidris_melanotos
run centronyx_henslowii


Executing:   0%|          | 0/23 [00:00<?, ?cell/s]

run buteo_lineatus
run anthus_spragueii
run coccyzus_americanus


...

metadata: 1: cells: not found
Executing:  96%|█████████▌| 22/23 [00:21<00:02,  2.19s/cell][IPKernelApp] ERROR | Invalid Control Message
Traceback (most recent call last):
  File "/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/site-packages/ipykernel/kernelbase.py", line 349, in process_control
    msg = self.session.deserialize(msg, content=True, copy=False)
  File "/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/site-packages/jupyter_client/session.py", line 1074, in deserialize
    raise ValueError(msg)
ValueError: Invalid Signature: b'991174f2f617377fe16a2467cc62c70a6d268520cc3b52ce9460e9322bd35794'
Executing:  22%|██▏       | 5/23 [00:04<00:11,  1.56cell/s]]Traceback (most recent call last):
  File "/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
                     "__main__", mod_spec)
  File "/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/runpy.py", line 88, 

run apalone_spinifera
run kinosternon_subrubrum


Executing:  39%|███▉      | 9/23 [00:05<00:05,  2.51cell/s]metadata: 1: cells: not found


run macrochelys_temmincki


Executing:  43%|████▎     | 10/23 [00:05<00:04,  2.75cell/s]Traceback (most recent call last):
  File "/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
                     "__main__", mod_spec)
  File "/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/runpy.py", line 88, in _run_code
    exec(code, run_globals)
    ~~~~^^^^^^^^^^^^^^^^^^^
  File "/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
    ~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/site-packages/traitlets/config/application.py", line 1074, in launch_instance
    app.initialize(argv)
    ~~~~~~~~~~~~~~^^^^^^
  File "/home/mike/miniforge3/envs/rapids-25.10/lib/python3.13/site-packages/traitlets/config/application.py", line 118, in inner
    return method(app, *args, **kwargs)
  File "/home

In [ ]:
print('## Failed species ##')
print('')
for i, result in enumerate(completed):
    if result[1] == 'fail':
        print(result[0])

## Failed species ##

{'name': 'protonotaria_citrea', 'group': 'avian'}
{'name': 'hyla_versicolor', 'group': 'herp'}
{'name': 'lithobates_palustris', 'group': 'herp'}
{'name': 'macrochelys_temmincki', 'group': 'herp'}
